Scenario: "The Environmental Policy Compliance Assistant"
Background
You are part of the Sustainability & Environmental Compliance Team at a global manufacturing company.
New government regulations on carbon emissions, waste disposal, and renewable energy adoption have just been released.
The company must ensure compliance to avoid fines and reputational damage.

Challenge
The company uploads a PDF of the environmental regulation into the Compliance Assistant (Gradio + Chroma + LLM app).
Your task is to:
- Process the regulation document so the assistant can store and understand it.
- Ask compliance-related questions about emission limits, waste management rules, and renewable energy targets.
- Generate clear, actionable answers that can guide engineers, sustainability officers, and executives.

Roles
- Learner (You): Environmental compliance officer using the assistant.
- Assistant (The RAG App): Provides answers strictly based on uploaded environmental regulations.
- Stakeholders: Plant managers, sustainability officers, and executives who need concise compliance guidance.

🔄 Flow of the Scenario
- Upload Environmental Regulation PDF
Example: “National Carbon Emissions Act 2026”.
- System Processes Document
- Splits into chunks.
- Embeds into vector database.
- Stores for retrieval.
- Ask Questions
- “What is the maximum carbon emission allowed per factory per year?”
- “What penalties apply if hazardous waste is not disposed of properly?”
- “What renewable energy targets must we meet by 2030?”
- Assistant Responds
- Retrieves relevant chunks.
- Generates compliance-focused answers.
- Provides short, clear guidance.
- Outcome
- Learners practice extracting environmental obligations.
- Managers receive summarized compliance insights.
- Executives gain confidence in sustainability strategy alignment.

🎯 Training Objective
This scenario helps learners:
- Understand how RAG systems can support environmental compliance.
- Practice formulating precise queries to extract obligations.
- Experience how AI can simplify complex environmental regulations into actionable steps.

👉 Would you like me to also draft a sample regulation PDF text (like the healthcare one I created earlier) for this environmental context, so you can upload it into your assistant and simulate queries?

STEP 1 — Install Dependencies

In [ ]:
!pip install gradio chromadb sentence-transformers pypdf transformers

STEP 2 — Import Libraries

In [2]:
import gradio as gr
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import chromadb
from transformers import pipeline

STEP 3 — Load Embedding Model

In [3]:
print("Loading embedding model...")

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded")

Loading embedding model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded


STEP 4 — Initialize Vector Database (Chroma)

In [4]:
client = chromadb.Client()

try:
    client.delete_collection("legal_docs")
except:
    pass

collection = client.create_collection("legal_docs")

STEP 5 — Load Language Model

In [5]:
print("Loading LLM...")

llm = pipeline(
    "question-answering",
    model="deepset/roberta-base-squad2"
)

print("LLM loaded successfully")

Loading LLM...


config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/496M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaForQuestionAnswering LOAD REPORT from: deepset/roberta-base-squad2
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/79.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

LLM loaded successfully


STEP 6 — Document Chunking

In [6]:
def chunk_text(text, chunk_size=500, overlap=50):

    chunks = []
    start = 0

    while start < len(text):

        end = start + chunk_size
        chunk = text[start:end]

        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks

STEP 7 — Process Uploaded PDF

In [7]:
def process_pdf(file):

    print("Processing PDF...")

    reader = PdfReader(file.name)

    text = ""

    for page in reader.pages:
        text += page.extract_text()

    chunks = chunk_text(text)

    print("Total chunks:", len(chunks))

    for i, chunk in enumerate(chunks):

        embedding = embedding_model.encode(chunk).tolist()

        collection.add(
            documents=[chunk],
            embeddings=[embedding],
            ids=[str(i)]
        )

    return f"Document processed successfully. {len(chunks)} chunks stored."

STEP 8 — Retriever

In [8]:
def retrieve(query, k=3):

    query_embedding = embedding_model.encode(query).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )

    docs = results["documents"][0]

    print("\nRetrieved Chunks:\n", docs)

    return docs

STEP 9 — Answer Generation

In [9]:
def answer_question(query):

    docs = retrieve(query)

    context = " ".join(docs)

    prompt = f"""
You are a legal compliance assistant.

Use ONLY the context below to answer the question.

Context:
{context}

Question: {query}

Provide a short clear answer.
"""

    response = llm(
       question=query,
    context=context
    )

    print("\nRaw Model Output:\n", response)

    return response['answer']

STEP 10 — Chat Function

In [10]:
def chat(question):

    if not question.strip():
        return "Please enter a question."

    answer = answer_question(question)

    if not answer:
        return "No answer generated."

    return answer

STEP 11 — Build Gradio Interface

In [11]:
with gr.Blocks() as demo:

    gr.Markdown("# 📜 Legal Compliance RAG Assistant")

    gr.Markdown("""
Upload a legal regulation document and ask questions about:

• compliance rules
• cross-border data transfers
• penalties for violations
• data subject rights
""")

    pdf_input = gr.File(label="Upload Legal PDF")

    upload_button = gr.Button("Process Document")

    status = gr.Textbox(label="Status")

    upload_button.click(
        process_pdf,
        inputs=pdf_input,
        outputs=status
    )

    question_box = gr.Textbox(
        label="Ask a Legal Question"
    )

    answer_box = gr.Textbox(
        label="Answer",
        lines=15
    )

    ask_button = gr.Button("Ask")

    ask_button.click(
        chat,
        inputs=question_box,
        outputs=answer_box
    )


STEP 12 — Launch Application

In [12]:
demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://422eb2fdfc2f772c1f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
